In [ ]:
#mdp
import numpy as np
states = []
for i in range(2**16):
    ith_state = []
    j = i
    while len(ith_state) < 16:
        ith_state.append(j % 2)
        j //= 2
    states.append(tuple(ith_state))  

terminal_state = tuple([0] * 16)

lo_map = {}

for state in states:
    lo_map[state] = {}

    for action in range(16):
        flipping = [action]

        if action - 4 >= 0:
            flipping.append(action - 4)
        if action + 4 < 16:
            flipping.append(action + 4)

        if action % 4 != 0:
            flipping.append(action - 1)
        if (action + 1) % 4 != 0:
            flipping.append(action + 1)

        new_state = list(state)
        for cell in flipping:
            new_state[cell] = 1 - new_state[cell]

        new_state = tuple(new_state)

        reward = 1 if new_state == terminal_state else 0

        lo_map[state][action] = [(1.0, new_state, reward, new_state == terminal_state)]

In [ ]:
#value Iteration
V = {s: 0.0 for s in states}
gamma = 0.9
theta = 1e-4

while True:
    delta = 0
    V_new = V.copy()

    for s in states:
        if s == terminal_state:
            continue

        best = float('-inf')

        for a in range(16):
            p, s_next, r, done = lo_map[s][a][0]
            val = p * (r + gamma * V[s_next])
            best = max(best, val)

        V_new[s] = best
        delta = max(delta, abs(V[s] - best))

    V = V_new
    if delta < theta:
        break


In [ ]:
#optimal policy
policy = {}

for s in states:
    if s == terminal_state:
        policy[s] = None
        continue

    best = float('-inf')
    best_action = None

    for a in range(16):
        p, s_next, r, done = lo_map[s][a][0]
        val = p * (r + gamma * V[s_next])

        if val > best:
            best = val
            best_action = a

    policy[s] = best_action